# Sinh voice cho SC-001–SC-040 bằng F5-TTS-Vietnamese + ViMD

Notebook **Google Colab GPU** đọc `calls.<call_id>.customer_turns` trong từng JSON,
chọn **một giọng ViMD đúng tỉnh và giới tính cho mỗi scenario**, dùng lại giọng đó
qua tất cả call, và xuất **một WAV cho từng phần tử** của `customer_turns`.
Không ghép các lượt hoặc các call. 40 JSON hiện tại có **80 call / 250 lượt**;
notebook tự đếm lại khi chạy, không hard-code số lượt.

**Cách chạy:**
1. Upload thư mục `scenarios_complete dataset` lên Google Drive.
2. Mở notebook trong Colab, chọn **Runtime → Change runtime type → GPU** (T4 hoặc mạnh hơn).
3. Chạy ô cài đặt, mount Drive, sửa `DATASET_DIR` và `OUTPUT_DIR` ở phần cấu hình.
4. Chạy phần kiểm tra dữ liệu và chọn reference, nghe các mẫu ở ô preview.
5. Chạy ô sinh WAV. `RUN_TTS=True` sẵn; đặt `MAX_TURNS=3` để thử trước,
   rồi `MAX_TURNS=None` và chạy lại để sinh đủ. Notebook nhận diện WAV đã hoàn thành.
6. Kết quả và cache reference được giữ trên Drive; checkpoint ở `/content` để suy luận nhanh.

Ví dụ: `audio/SC-001/call_1/SC-001_call_1_turn_001.wav`,
`...turn_002.wav`, `...turn_003.wav` tương ứng đúng thứ tự trong list.
Manifest có cả **index gốc 0-based** và **số thứ tự 1-based**, câu gốc, câu đọc,
giọng mẫu, seed, thời lượng và trạng thái.

Notebook tham khảo `customer_turns_to_f5_tts_vietnamese.ipynb` và dùng cùng
checkpoint [`hynt/F5-TTS-Vietnamese-ViVoice`](https://huggingface.co/hynt/F5-TTS-Vietnamese-ViVoice).
Nguồn giọng: [`nguyendv02/ViMD_Dataset`](https://huggingface.co/datasets/nguyendv02/ViMD_Dataset).
Tỉnh và giới tính là tiêu chí chọn **reference**; cần nghe kiểm tra accent của audio sinh ra,
vì F5-TTS không bảo đảm tái tạo chính xác mọi phương ngữ. Metadata vùng yêu cầu và vùng ViMD
được lưu riêng. SC-028 “Hà Tỉnh” được chuẩn hóa tên thành **Hà Tĩnh**, giữ nhãn yêu cầu **Miền bắc**.

Nguồn kỹ thuật: [F5-TTS-Vietnamese](https://github.com/nguyenthienhy/F5-TTS-Vietnamese),
[bài báo ViMD](https://aclanthology.org/2024.emnlp-main.426/).
Thông tin giấy phép nguồn: ViMD **CC BY-NC-ND 4.0**, checkpoint **CC BY-NC-SA 4.0**;
notebook giữ lại thông tin này trong manifest.

## 1. Cài đặt

Dùng dependency `f5-tts==1.1.22` như notebook mẫu, nhưng chạy source fork Vietnamese
tại commit cố định. Cài fork với `--no-deps` để không kéo pin NumPy cũ `<=1.26.4`
vào Colab. Không cài lại bộ Torch/CUDA đang có của Colab nếu đã đáp ứng dependency.
Nếu ô này yêu cầu restart, chọn **Runtime → Restart session**, rồi chạy lại từ đầu.

In [ ]:
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

F5_REPO_URL = 'https://github.com/nguyenthienhy/F5-TTS-Vietnamese.git'
F5_REPO_COMMIT = 'e74db9d5a5e521bb930490e7cd7912438bf7ae84'
F5_REPO_DIR = Path('/content/F5-TTS-Vietnamese')
if not F5_REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--no-checkout', F5_REPO_URL, str(F5_REPO_DIR)], check=True)
    subprocess.run(['git', '-C', str(F5_REPO_DIR), 'checkout', '--detach', F5_REPO_COMMIT], check=True)
else:
    actual_commit = subprocess.check_output(
        ['git', '-C', str(F5_REPO_DIR), 'rev-parse', 'HEAD'], text=True
    ).strip()
    if actual_commit != F5_REPO_COMMIT:
        raise RuntimeError('Thư mục F5 trong /content có commit khác. Dùng một Colab runtime mới.')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', 'f5-tts==1.1.22',
    'jieba', 'pandas', 'pyarrow', 'fsspec', 'huggingface_hub',
    'soundfile', 'librosa', 'tqdm'
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(F5_REPO_DIR)
], check=True)

src = str(F5_REPO_DIR / 'src')
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()
if 'f5_tts' in sys.modules:
    raise RuntimeError('F5 đã được import trước khi cài source. Restart session rồi chạy lại.')
if shutil.which('ffmpeg') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)

probe = subprocess.run([
    sys.executable, '-c',
    'import numpy, scipy, torch, torchaudio, soundfile; from f5_tts.api import F5TTS; print(numpy.__version__, torch.__version__)'
], capture_output=True, text=True)
if probe.returncode:
    print(probe.stderr[-4000:])
    raise RuntimeError('Dependency chưa import được. Restart session rồi chạy lại ô này; xem lỗi bên trên nếu vẫn lỗi.')
print('Dependency:', probe.stdout.strip())
print('Source F5:', F5_REPO_COMMIT)

## 2. Mount Drive và cấu hình

`SCENARIO_IDS=None` = đủ 40 scenario đã chỉ định; có thể dùng `['SC-001']` để thử.
`CALL_IDS=None` = tất cả call trong mỗi scenario. `MAX_TURNS` giới hạn tổng số lượt cho một lần chạy.
`OVERWRITE=False` vẫn tự sinh lại file nếu câu đọc, reference hoặc cấu hình model đã đổi.
Đặt `OVERWRITE=True` khi muốn chủ động sinh lại mọi lượt được chọn.

ViMD được đọc từ **Parquet gốc**, vì Dataset Viewer có thể chỉ index một phần dataset lớn.
Phần metadata chỉ đọc cột text/nhãn. Khi lấy audio, Parquet tải **cột audio của row group**
chứa ứng viên (có thể vài trăm MB mỗi group), rồi chỉ lưu WAV reference được chọn.
Lần đầu lấy đủ 40 giọng có thể truyền nhiều GB; các lần sau dùng cache trên Drive.
Không tải toàn bộ gần 60 GB dataset xuống ổ đĩa Colab.

Mặc định chỉ dùng split `train`. Không tự đổi tỉnh/giới tính nếu thiếu mẫu.
`REFERENCE_FILENAME_OVERRIDES` cho phép chọn lại bản ghi sau khi nghe:
ví dụ `{'SC-001': '40_0123.wav'}` — phải tồn tại và vẫn đúng tỉnh/giới tính.
Đổi override rồi chạy lại các ô reference và TTS; các WAV phụ thuộc sẽ được cập nhật.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR = Path('/content/drive/MyDrive/scenarios_complete dataset')
OUTPUT_DIR = Path('/content/drive/MyDrive/tts_data/f5_vimd_SC001_SC040')
MODEL_CACHE_DIR = Path('/content/f5_vietnamese_model')

SCENARIO_IDS = None
CALL_IDS = None
MAX_TURNS = None
RUN_TTS = True
OVERWRITE = False

VIMD_DATASET_ID = 'nguyendv02/ViMD_Dataset'
VIMD_REVISION = '3a5b30157034e7eadd5c75fae1a820c6f9383398'
REFERENCE_SPLITS = ['train']  # Có thể chủ động thêm 'valid', 'test'.
REFERENCE_FILENAME_OVERRIDES = {}
REF_MIN_SECONDS = 3.0
REF_MAX_SECONDS = 11.0  # Dưới ngưỡng 12 giây mà fork có thể cắt reference.
REF_TARGET_SECONDS = 7.0
MAX_REFERENCE_GROUPS = 4

MODEL_REPO_ID = 'hynt/F5-TTS-Vietnamese-ViVoice'
MODEL_REVISION = '5c14cdd4e3f62b69e75140558bea2b2e293ee8f1'
MODEL_NAME = 'F5TTS_Base'
SPEED = 1.0
NFE_STEP = 32
CFG_STRENGTH = 2.0
SWAY_SAMPLING_COEF = -1.0
BASE_SEED = 20261007

# Chỉ áp dụng cho câu cần sinh, không áp dụng cho transcript reference.
NORMALIZE_NUMBERS = True
PRONUNCIATION_REPLACEMENTS = {'COD': 'si ô đi'}  # Thêm tên sản phẩm nếu cần.
TEXT_OVERRIDES = {}  # Ví dụ {'SC-001/call_1/0': 'Câu đọc chỉnh thủ công ...'}.

PREVIEW_SCENARIO_IDS = ['SC-001', 'SC-002', 'SC-028']
EXPORT_ZIP = False  # Bật ở cuối để đóng gói WAV + metadata + reference.

if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f'Không thấy thư mục JSON: {DATASET_DIR}. Sửa DATASET_DIR trước.')
if SPEED <= 0 or NFE_STEP <= 0:
    raise ValueError('SPEED và NFE_STEP phải > 0.')
if not 0 < REF_MIN_SECONDS <= REF_MAX_SECONDS < 12:
    raise ValueError('Reference phải nằm trong khoảng (0, 12) giây.')
if MAX_TURNS is not None and (not isinstance(MAX_TURNS, int) or MAX_TURNS <= 0):
    raise ValueError('MAX_TURNS phải là số nguyên > 0 hoặc None.')
if MAX_REFERENCE_GROUPS < 1:
    raise ValueError('MAX_REFERENCE_GROUPS phải >= 1.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('JSON:', DATASET_DIR)
print('Output:', OUTPUT_DIR)

## 3. Danh sách giọng theo yêu cầu

In [ ]:
import gc
import hashlib
import io
import json
import os
import re
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch
from huggingface_hub import HfFileSystem, hf_hub_download
from IPython.display import Audio, display
from tqdm.auto import tqdm

VOICE_ROWS = [
    ('SC-001', 'Nữ', 'Nghệ An', 'Miền trung'),
    ('SC-002', 'Nam', 'Cao Bằng', 'Miền bắc'),
    ('SC-003', 'Nữ', 'Quảng Bình', 'Miền trung'),
    ('SC-004', 'Nam', 'Đà Nẵng', 'Miền trung'),
    ('SC-005', 'Nữ', 'Quảng Ngãi', 'Miền trung'),
    ('SC-006', 'Nam', 'Thái Bình', 'Miền bắc'),
    ('SC-007', 'Nữ', 'Bình Thuận', 'Miền trung'),
    ('SC-008', 'Nam', 'Hà Giang', 'Miền bắc'),
    ('SC-009', 'Nữ', 'Bình Phước', 'Miền nam'),
    ('SC-010', 'Nam', 'Bà Rịa Vũng Tàu', 'Miền nam'),
    ('SC-011', 'Nữ', 'Hồ Chí Minh', 'Miền nam'),
    ('SC-012', 'Nam', 'Hà Nội', 'Miền bắc'),
    ('SC-013', 'Nữ', 'Quảng Ninh', 'Miền bắc'),
    ('SC-014', 'Nam', 'Bình Dương', 'Miền nam'),
    ('SC-015', 'Nữ', 'Gia Lai', 'Miền trung'),
    ('SC-016', 'Nam', 'Thái Nguyên', 'Miền bắc'),
    ('SC-017', 'Nữ', 'Ninh Thuận', 'Miền trung'),
    ('SC-018', 'Nam', 'Lào Cai', 'Miền bắc'),
    ('SC-019', 'Nữ', 'Phú Thọ', 'Miền bắc'),
    ('SC-020', 'Nam', 'Hậu Giang', 'Miền nam'),
    ('SC-021', 'Nữ', 'Long An', 'Miền nam'),
    ('SC-022', 'Nam', 'Sơn La', 'Miền bắc'),
    ('SC-023', 'Nữ', 'Bình Định', 'Miền trung'),
    ('SC-024', 'Nam', 'Đắk Lắk', 'Miền trung'),
    ('SC-025', 'Nữ', 'Hải Phòng', 'Miền bắc'),
    ('SC-026', 'Nam', 'Kon Tum', 'Miền trung'),
    ('SC-027', 'Nữ', 'Phú Yên', 'Miền trung'),
    ('SC-028', 'Nam', 'Hà Tĩnh', 'Miền bắc'),
    ('SC-029', 'Nữ', 'Đắk Nông', 'Miền trung'),
    ('SC-030', 'Nam', 'Lạng Sơn', 'Miền bắc'),
    ('SC-031', 'Nữ', 'Thừa Thiên Huế', 'Miền trung'),
    ('SC-032', 'Nam', 'Lâm Đồng', 'Miền trung'),
    ('SC-033', 'Nữ', 'Tây Ninh', 'Miền nam'),
    ('SC-034', 'Nam', 'Cà Mau', 'Miền nam'),
    ('SC-035', 'Nữ', 'Nam Định', 'Miền bắc'),
    ('SC-036', 'Nam', 'Thanh Hóa', 'Miền trung'),
    ('SC-037', 'Nữ', 'Lai Châu', 'Miền bắc'),
    ('SC-038', 'Nam', 'Trà Vinh', 'Miền nam'),
    ('SC-039', 'Nữ', 'Quảng Trị', 'Miền trung'),
    ('SC-040', 'Nam', 'Quảng Nam', 'Miền trung'),
]

def province_key(value):
    value = str(value).replace('Đ', 'D').replace('đ', 'd')
    value = unicodedata.normalize('NFKD', value)
    return re.sub('[^a-z]', '', ''.join(c for c in value if not unicodedata.combining(c)).lower())

VOICE_SPECS = {
    sid: {'scenario_id': sid, 'gender_label': gender, 'gender': 0 if gender == 'Nữ' else 1,
          'province_requested': province, 'province_key': province_key(province),
          'region_requested': region}
    for sid, gender, province, region in VOICE_ROWS
}
if SCENARIO_IDS is None:
    SELECTED_IDS = list(VOICE_SPECS)
else:
    if not isinstance(SCENARIO_IDS, list) or not SCENARIO_IDS:
        raise ValueError('SCENARIO_IDS phải là list không rỗng hoặc None.')
    SELECTED_IDS = list(dict.fromkeys(SCENARIO_IDS))
    missing = set(SELECTED_IDS) - set(VOICE_SPECS)
    if missing:
        raise ValueError(f'Chỉ cấu hình SC-001 đến SC-040; ID không hợp lệ: {missing}')
if CALL_IDS is not None and (not isinstance(CALL_IDS, list) or not CALL_IDS):
    raise ValueError('CALL_IDS phải là list không rỗng hoặc None.')
display(pd.DataFrame([VOICE_SPECS[sid] for sid in SELECTED_IDS]).drop(columns='province_key'))

## 4. Đọc JSON và xem trước từng lượt

Chỉ lấy `customer_turns`. Các trường `notes`, `facts_established`, `success_if`, ...
là dữ liệu kịch bản, không được đưa vào TTS hay thực hiện như chỉ thị.
Nếu có lượt rỗng/sai kiểu, dừng kiểm tra để tránh làm dịch index.

Mặc định chuẩn hóa số tiền, ngày ISO, ngày `dd/mm`, giờ `hh:mm`, số điện thoại và số nguyên
thành cách đọc tiếng Việt; câu gốc luôn được giữ trong manifest. `TEXT_OVERRIDES` dùng khóa
`scenario_id/call_id/turn_index` (index **0-based**) cho cách đọc riêng; đặt
`NORMALIZE_NUMBERS=False` và `PRONUNCIATION_REPLACEMENTS={}` nếu muốn đưa nguyên văn vào model.

In [ ]:
DIGITS = ['không', 'một', 'hai', 'ba', 'bốn', 'năm', 'sáu', 'bảy', 'tám', 'chín']

def read_triplet(number, padded=False):
    hundred, rest = divmod(number, 100)
    ten, unit = divmod(rest, 10)
    words = []
    if hundred or padded:
        words += [DIGITS[hundred], 'trăm']
    if ten >= 2:
        words += [DIGITS[ten], 'mươi']
    elif ten == 1:
        words += ['mười']
    elif unit and (hundred or padded):
        words += ['linh']
    if unit:
        words += ['mốt' if unit == 1 and ten >= 2 else 'lăm' if unit == 5 and ten >= 1 else DIGITS[unit]]
    return ' '.join(words)

def read_integer(number):
    number = int(number)
    if number == 0:
        return DIGITS[0]
    if number < 0:
        return 'âm ' + read_integer(-number)
    units = ['', 'nghìn', 'triệu', 'tỷ', 'nghìn tỷ', 'triệu tỷ']
    groups = []
    while number:
        number, remainder = divmod(number, 1000)
        groups.append(remainder)
    if len(groups) > len(units):
        raise ValueError('Số vượt giới hạn bộ đọc số; dùng TEXT_OVERRIDES.')
    result = []
    for i in reversed(range(len(groups))):
        if groups[i]:
            result.append(read_triplet(groups[i], padded=bool(result) and groups[i] < 100))
            if units[i]:
                result.append(units[i])
    return ' '.join(result)

def normalize_spoken_text(text):
    text = unicodedata.normalize('NFC', text).strip()
    if NORMALIZE_NUMBERS:
        # Thay ngày trước số để không mất cấu trúc ngày tháng.
        text = re.sub(r'\b(\d{4})-(\d{2})-(\d{2})\b',
                      lambda m: f'{read_integer(m[3])} tháng {read_integer(m[2])} năm {read_integer(m[1])}', text)
        text = re.sub(r'\b(\d{1,2})/(\d{1,2})(?:/(\d{4}))?\b',
                      lambda m: f'{read_integer(m[1])} tháng {read_integer(m[2])}' +
                      (f' năm {read_integer(m[3])}' if m[3] else ''), text)
        text = re.sub(r'\b(\d{1,2}):(\d{2})\b',
                      lambda m: f'{read_integer(m[1])} giờ' +
                      (f' {read_integer(m[2])} phút' if int(m[2]) else ''), text)
        text = re.sub(r'(?<!\d)\d{1,3}(?:\.\d{3})+(?!\d)',
                      lambda m: read_integer(m[0].replace('.', '')), text)
        text = re.sub(r'(?<!\w)0\d{8,10}(?!\w)',
                      lambda m: ' '.join(DIGITS[int(d)] for d in m[0]), text)
        text = re.sub(r'\d+,\d+', lambda m: read_integer(m[0].split(',')[0]) +
                      ' phẩy ' + ' '.join(DIGITS[int(d)] for d in m[0].split(',')[1]), text)
        text = re.sub(r'\d+', lambda m: read_integer(m[0]), text)
    for source, target in PRONUNCIATION_REPLACEMENTS.items():
        text = re.sub(r'(?<!\w)' + re.escape(source) + r'(?!\w)', target, text, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).lower().strip()

def safe_component(value):
    # Không cho tên call trong JSON ghi ra ngoài thư mục output.
    value = str(value)
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', value):
        raise ValueError(f'ID không an toàn để đặt tên file: {value!r}')
    return value

def build_tasks(dataset_dir, selected_ids):
    tasks = []
    for sid in selected_ids:
        source_file = dataset_dir / f'{sid}.json'
        payload = json.loads(source_file.read_text(encoding='utf-8-sig'))
        if not isinstance(payload, dict) or payload.get('scenario_id', sid) != sid:
            raise ValueError(f'{source_file.name}: scenario_id không khớp tên file.')
        calls = payload.get('calls')
        if not isinstance(calls, dict) or not calls:
            raise ValueError(f'{sid}: thiếu calls hoặc calls rỗng.')
        chosen_calls = list(calls) if CALL_IDS is None else list(dict.fromkeys(CALL_IDS))
        for call_id in chosen_calls:
            safe_component(call_id)
            if call_id not in calls or not isinstance(calls[call_id], dict):
                raise ValueError(f'{sid}: call không hợp lệ: {call_id}')
            turns = calls[call_id].get('customer_turns')
            if not isinstance(turns, list) or not turns:
                raise ValueError(f'{sid}/{call_id}: customer_turns phải là list không rỗng.')
            for index, original in enumerate(turns):
                if not isinstance(original, str) or not original.strip():
                    raise ValueError(f'{sid}/{call_id}/index={index}: lượt rỗng/sai kiểu; không tự bỏ để giữ index.')
                task_id = f'{sid}/{call_id}/{index}'
                spoken = TEXT_OVERRIDES.get(task_id, normalize_spoken_text(original))
                if not isinstance(spoken, str) or not spoken.strip():
                    raise ValueError(f'TEXT_OVERRIDES rỗng/sai kiểu: {task_id}')
                relative = f'audio/{sid}/{call_id}/{sid}_{call_id}_turn_{index + 1:03d}.wav'
                tasks.append({'task_id': task_id, 'scenario_id': sid, 'call_id': call_id,
                              'turn_index': index, 'turn_number': index + 1,
                              'source_file': str(source_file), 'text_original': original,
                              'text_tts': spoken, 'output_relative': relative})
    return tasks

TASKS = build_tasks(DATASET_DIR, SELECTED_IDS)
summary = pd.DataFrame(TASKS).groupby('scenario_id').agg(
    calls=('call_id', 'nunique'), customer_turns=('task_id', 'count'))
display(summary)
print(f'Tổng: {len(SELECTED_IDS)} scenario / {summary.calls.sum()} call / {len(TASKS)} file WAV.')
display(pd.DataFrame(TASKS)[['task_id', 'text_original', 'text_tts', 'output_relative']].head(8))

## 5. Đọc metadata ViMD, không đọc cột audio

Cache được gắn revision và split. Mỗi ứng viên lưu shard, row group và index trong group
để lấy đúng audio về sau. Chuẩn hóa dấu/viết hoa cho tên tỉnh (`Nghệ An` ↔ `NgheAn`).
Lần chạy đầu có thể cần vài phút; lỗi mạng sẽ báo rõ, chạy lại ô để thử lại.

In [ ]:
def atomic_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    for attempt in range(5):
        try:
            temporary.replace(path)
            break
        except PermissionError:
            if attempt == 4:
                raise
            time.sleep(0.1 * (attempt + 1))

def fingerprint(data):
    return hashlib.sha256(json.dumps(data, ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

HF_FS = HfFileSystem(token=os.getenv('HF_TOKEN'))
METADATA_DIR = OUTPUT_DIR / 'cache'
METADATA_DIR.mkdir(parents=True, exist_ok=True)
METADATA_FILE = METADATA_DIR / 'vimd_metadata.parquet'
METADATA_CONFIG_FILE = METADATA_DIR / 'vimd_metadata_config.json'
metadata_config = {'version': 1, 'dataset_id': VIMD_DATASET_ID,
                   'revision': VIMD_REVISION, 'splits': REFERENCE_SPLITS}

def load_vimd_metadata():
    if METADATA_FILE.is_file() and METADATA_CONFIG_FILE.is_file():
        if json.loads(METADATA_CONFIG_FILE.read_text(encoding='utf-8')) == metadata_config:
            print('Dùng cache metadata:', METADATA_FILE)
            return pd.read_parquet(METADATA_FILE)
    if not REFERENCE_SPLITS or set(REFERENCE_SPLITS) - {'train', 'valid', 'test'}:
        raise ValueError('REFERENCE_SPLITS chỉ nhận train, valid, test và không được rỗng.')
    root = f'datasets/{VIMD_DATASET_ID}@{VIMD_REVISION}'
    paths = sorted(HF_FS.glob(f'{root}/data/*.parquet'))
    paths = [p for p in paths if p.rsplit('/', 1)[-1].split('-', 1)[0] in REFERENCE_SPLITS]
    if not paths:
        raise FileNotFoundError('Không thấy Parquet gốc ở revision/split đã chọn.')
    columns = ['region', 'province_code', 'province_name', 'filename', 'text', 'speakerID', 'gender']
    def read_metadata_shard(path):
        shard_frames = []
        with HF_FS.open(path, 'rb', block_size=65536, cache_type='readahead') as stream:
            parquet = pq.ParquetFile(stream)
            missing = set(columns + ['audio']) - set(parquet.schema_arrow.names)
            if missing:
                raise ValueError(f'Schema ViMD thiếu cột: {missing}')
            for group_index in range(parquet.num_row_groups):
                frame = parquet.read_row_group(group_index, columns=columns).to_pandas()
                frame['source_parquet'] = path
                frame['row_group'] = group_index
                frame['row_in_group'] = np.arange(len(frame))
                frame['split'] = path.rsplit('/', 1)[-1].split('-', 1)[0]
                shard_frames.append(frame)
        return shard_frames
    frames = []
    with ThreadPoolExecutor(max_workers=4) as pool:
        for shard_frames in tqdm(pool.map(read_metadata_shard, paths), total=len(paths),
                                 desc='Metadata ViMD (không audio)'):
            frames.extend(shard_frames)
    metadata = pd.concat(frames, ignore_index=True)
    temp = METADATA_FILE.with_name(METADATA_FILE.name + '.partial')
    metadata.to_parquet(temp, index=False)
    temp.replace(METADATA_FILE)
    atomic_json(METADATA_CONFIG_FILE, metadata_config)
    return metadata

VIMD_METADATA = load_vimd_metadata()
VIMD_METADATA['province_key'] = VIMD_METADATA.province_name.map(province_key)
VIMD_METADATA['word_count'] = VIMD_METADATA.text.fillna('').str.split().str.len()
VIMD_METADATA['gender'] = pd.to_numeric(VIMD_METADATA.gender, errors='raise').astype(int)

coverage = []
for sid in SELECTED_IDS:
    spec = VOICE_SPECS[sid]
    matches = VIMD_METADATA[(VIMD_METADATA.province_key == spec['province_key']) &
                            (VIMD_METADATA.gender == spec['gender'])]
    coverage.append({'scenario_id': sid, 'province_requested': spec['province_requested'],
                     'gender_requested': spec['gender_label'], 'candidates': len(matches),
                     'province_vimd': ', '.join(sorted(matches.province_name.unique())),
                     'region_vimd': ', '.join(sorted(matches.region.unique()))})
COVERAGE = pd.DataFrame(coverage)
display(COVERAGE)
if (COVERAGE.candidates == 0).any():
    missing = COVERAGE.loc[COVERAGE.candidates == 0, 'scenario_id'].tolist()
    raise ValueError(f'Không có giọng đúng tỉnh/giới tính: {missing}. Kiểm tra tên tỉnh hoặc thêm split; không dùng giọng khác thay thế.')
print(f'Metadata: {len(VIMD_METADATA):,} bản ghi; đủ tỉnh/giới tính cho {len(SELECTED_IDS)} scenario.')

## 6. Chọn và lưu một reference cho mỗi scenario

Ưu tiên transcript khoảng 24 từ, audio nguyên bản 3–11 giây (mục tiêu 7 giây),
không im lặng và không clipping nặng. Đọc mono và resample về 24 kHz nếu cần.
**Không cắt lời, không ghép audio, không tự nhận dạng lại transcript**;
`ref_text` lấy từ metadata đúng bản ghi. Kiểm tra tín hiệu chỉ là sàng lọc cơ bản;
hãy nghe để nhận biết nhạc nền, nhiều người nói hoặc transcript sai.

Chỉ giữ một audio row group trong RAM mỗi lần. Nếu không tìm được mẫu hợp lệ,
notebook ghi lỗi và dừng trước khi sinh hàng loạt. Có thể tăng `MAX_REFERENCE_GROUPS`
hoặc chỉ định một `REFERENCE_FILENAME_OVERRIDES` khác rồi chạy lại.

In [ ]:
REFERENCE_DIR = OUTPUT_DIR / 'references'
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_BANK_FILE = REFERENCE_DIR / 'reference_bank.json'

class AudioGroupReader:
    def __init__(self, filesystem):
        self.filesystem = filesystem
        self.key = None
        self.table = None

    def get(self, row):
        key = (row['source_parquet'], int(row['row_group']))
        if key != self.key:
            self.table = None
            self.key = None
            gc.collect()
            with self.filesystem.open(key[0], 'rb', block_size=65536, cache_type='none') as stream:
                self.table = pq.ParquetFile(stream).read_row_group(key[1], columns=['audio'])
            self.key = key
        item = self.table.column('audio')[int(row['row_in_group'])].as_py()
        raw_bytes = item.get('bytes') if isinstance(item, dict) else None
        if not raw_bytes:
            raise ValueError('Audio không có embedded bytes; không tự dùng đường dẫn khác.')
        waveform, sample_rate = sf.read(io.BytesIO(raw_bytes), dtype='float32', always_2d=True)
        return waveform.mean(axis=1), int(sample_rate)

def audio_metrics(waveform, sample_rate):
    if not len(waveform) or sample_rate <= 0 or not np.isfinite(waveform).all():
        raise ValueError('Audio rỗng hoặc có NaN/Inf.')
    return {'duration_seconds': float(len(waveform) / sample_rate),
            'rms': float(np.sqrt(np.mean(waveform ** 2))),
            'peak': float(np.max(np.abs(waveform))),
            'clipping_fraction': float(np.mean(np.abs(waveform) >= 0.999))}

def reference_request(spec):
    return {'dataset_id': VIMD_DATASET_ID, 'revision': VIMD_REVISION,
            'splits': REFERENCE_SPLITS, 'province_key': spec['province_key'],
            'gender': spec['gender'], 'filename_override': REFERENCE_FILENAME_OVERRIDES.get(spec['scenario_id']),
            'min_seconds': REF_MIN_SECONDS, 'max_seconds': REF_MAX_SECONDS,
            'target_seconds': REF_TARGET_SECONDS, 'selection_version': 1}

def choose_reference(spec, reader):
    sid = spec['scenario_id']
    pool = VIMD_METADATA[(VIMD_METADATA.province_key == spec['province_key']) &
                         (VIMD_METADATA.gender == spec['gender'])].copy()
    pool = pool[pool.text.notna() & pool.text.str.strip().ne('') & pool.speakerID.notna()]
    override = REFERENCE_FILENAME_OVERRIDES.get(sid)
    if override:
        pool = pool[pool.filename == override]
        if pool.empty:
            raise ValueError(f'{sid}: override {override} không đúng tỉnh/giới tính/split.')
    pool['distance'] = (pool.word_count - 24).abs()
    pool = pool.sort_values(['distance', 'filename', 'split'], kind='stable')
    groups = pool[['source_parquet', 'row_group']].drop_duplicates().head(MAX_REFERENCE_GROUPS)
    rejected = []
    for group in groups.to_dict('records'):
        group_pool = pool[(pool.source_parquet == group['source_parquet']) &
                          (pool.row_group == group['row_group'])]
        # Xét các câu ngắn trong cùng group đã tải để tránh đọc lại hàng trăm MB.
        group_pool = group_pool if override else group_pool[group_pool.word_count.between(5, 50)]
        valid = []
        for row in group_pool.to_dict('records'):
            waveform, sample_rate = reader.get(row)
            metrics = audio_metrics(waveform, sample_rate)
            if not REF_MIN_SECONDS <= metrics['duration_seconds'] <= REF_MAX_SECONDS:
                rejected.append(f"{row['filename']}: {metrics['duration_seconds']:.2f}s")
                continue
            if metrics['rms'] < 0.005 or metrics['clipping_fraction'] > 0.01:
                rejected.append(f"{row['filename']}: im lặng/clipping")
                continue
            score = abs(metrics['duration_seconds'] - REF_TARGET_SECONDS)
            valid.append((score, row, waveform, sample_rate, metrics))
        if not valid:
            continue
        _, row, waveform, source_sr, metrics = min(valid, key=lambda item: (item[0], item[1]['filename']))
        if source_sr != 24000:
            import librosa
            waveform = librosa.resample(waveform, orig_sr=source_sr, target_sr=24000)
        reference_path = REFERENCE_DIR / sid / 'reference.wav'
        reference_path.parent.mkdir(parents=True, exist_ok=True)
        temporary = reference_path.with_name('reference.partial.wav')
        sf.write(temporary, waveform, 24000, subtype='PCM_16')
        temporary.replace(reference_path)
        record = {key: row[key] for key in ['split', 'province_name', 'region', 'filename', 'speakerID', 'source_parquet']}
        record.update({'scenario_id': sid, 'gender': int(row['gender']),
                       'province_code': int(row['province_code']),
                       'row_group': int(row['row_group']), 'row_in_group': int(row['row_in_group']),
                       'dataset_id': VIMD_DATASET_ID, 'dataset_revision': VIMD_REVISION,
                       'ref_text_original': row['text'], 'ref_text_tts': row['text'].strip().lower(),
                       'ref_audio_relative': reference_path.relative_to(OUTPUT_DIR).as_posix(),
                       'source_sample_rate': source_sr, 'sample_rate': 24000,
                       'audio_sha256': file_sha256(reference_path), 'metrics': metrics,
                       'request_fingerprint': fingerprint(reference_request(spec))})
        return record
    raise ValueError(f'{sid}: chưa tìm thấy reference nguyên bản hợp lệ trong {len(groups)} group. '
                     f'Mẫu bị loại: {rejected[:12]}. Chọn filename khác hoặc tăng MAX_REFERENCE_GROUPS.')

def prepare_references():
    bank = json.loads(REFERENCE_BANK_FILE.read_text(encoding='utf-8')) if REFERENCE_BANK_FILE.is_file() else {}
    reader = AudioGroupReader(HF_FS)
    errors = []
    for sid in tqdm(SELECTED_IDS, desc='Chọn giọng reference'):
        spec = VOICE_SPECS[sid]
        existing = bank.get(sid)
        cached_audio = OUTPUT_DIR / existing['ref_audio_relative'] if existing else None
        if (existing and existing.get('request_fingerprint') == fingerprint(reference_request(spec))
                and cached_audio.is_file() and file_sha256(cached_audio) == existing.get('audio_sha256')):
            continue
        try:
            bank[sid] = choose_reference(spec, reader)
            atomic_json(REFERENCE_BANK_FILE, bank)
            print(f"{sid}: {bank[sid]['filename']} / {bank[sid]['speakerID']} / {bank[sid]['metrics']['duration_seconds']:.2f}s")
        except Exception as exc:
            errors.append({'scenario_id': sid, 'error': f'{type(exc).__name__}: {exc}'})
            print(errors[-1])
    reader.table = None
    gc.collect()
    atomic_json(REFERENCE_DIR / 'reference_errors.json', errors)
    if errors:
        raise RuntimeError(f'{len(errors)} scenario chưa có reference đúng cấu hình. Xem references/reference_errors.json rồi chạy lại ô này.')
    return {sid: bank[sid] for sid in SELECTED_IDS}

REFERENCES = {}  # Xóa reference trong RAM của lần chạy trước nếu lần chọn mới bị lỗi.
REFERENCES = prepare_references()
reference_summary = pd.DataFrame([
    {'scenario_id': sid, 'gender': VOICE_SPECS[sid]['gender_label'],
     'province_requested': VOICE_SPECS[sid]['province_requested'],
     'region_requested': VOICE_SPECS[sid]['region_requested'],
     'province_vimd': ref['province_name'], 'region_vimd': ref['region'],
     'speakerID': ref['speakerID'], 'filename': ref['filename'],
     'duration_seconds': ref['metrics']['duration_seconds']}
    for sid, ref in REFERENCES.items()
])
reference_summary.to_csv(REFERENCE_DIR / 'reference_summary.csv', index=False, encoding='utf-8-sig')
display(reference_summary)

## 7. Nghe thử giọng mẫu và xem transcript

In [ ]:
for sid in PREVIEW_SCENARIO_IDS:
    if sid not in REFERENCES:
        continue
    ref = REFERENCES[sid]
    print(f"{sid} | {VOICE_SPECS[sid]['gender_label']} | {ref['province_name']} | {ref['speakerID']}")
    print(ref['ref_text_original'])
    display(Audio(filename=str(OUTPUT_DIR / ref['ref_audio_relative'])))

## 8. Nạp checkpoint lên GPU (một lần)

`config.json` của checkpoint là **vocabulary dạng text**, không phải JSON cấu hình model;
lưu thành `vocab.txt` như model card. Dùng kiến trúc `F5TTS_Base` của notebook gốc.
Giữ engine trong RAM khi chạy lại ô generation; không tải model riêng cho từng scenario.

In [ ]:
def load_engine():
    if not torch.cuda.is_available():
        raise RuntimeError('Chưa có GPU CUDA. Chọn Runtime → Change runtime type → GPU.')
    from f5_tts.api import F5TTS
    MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint = Path(hf_hub_download(MODEL_REPO_ID, 'model_last.pt',
                                    revision=MODEL_REVISION, local_dir=MODEL_CACHE_DIR))
    vocabulary_source = Path(hf_hub_download(MODEL_REPO_ID, 'config.json',
                                           revision=MODEL_REVISION, local_dir=MODEL_CACHE_DIR))
    vocabulary = MODEL_CACHE_DIR / 'vocab.txt'
    vocabulary.write_bytes(vocabulary_source.read_bytes())
    print('GPU:', torch.cuda.get_device_name(0))
    print('Checkpoint:', checkpoint)
    return F5TTS(model=MODEL_NAME, ckpt_file=str(checkpoint), vocab_file=str(vocabulary), device='cuda')

if RUN_TTS:
    engine_config_key = fingerprint({'model': MODEL_REPO_ID, 'revision': MODEL_REVISION, 'name': MODEL_NAME,
                                     'fork': F5_REPO_COMMIT})
    if globals().get('_ENGINE_CONFIG_KEY') != engine_config_key or globals().get('ENGINE') is None:
        ENGINE = load_engine()
        _ENGINE_CONFIG_KEY = engine_config_key
    else:
        print('Dùng lại engine trên GPU.')
else:
    print('RUN_TTS=False: đã chuẩn bị dữ liệu/reference, chưa nạp checkpoint.')

## 9. Sinh một WAV cho mỗi lượt và lưu tiến độ

WAV PCM 16-bit mono, sample rate của engine (thông thường 24 kHz). Một lượt có thể được
F5 chunk nội bộ khi dài, nhưng chỉ xuất **một file** cho toàn bộ lượt đó.
Đánh index lại từ 001 trong từng call, luôn kèm scenario và call để không trùng tên.

Sau mỗi lượt, ghi `manifest.json` nguyên tử để chạy tiếp được sau khi ngắt phiên.
Chỉ bỏ qua file có trạng thái thành công, cùng fingerprint đầu vào và WAV có checksum khớp.
Lỗi một lượt không ngăn các lượt sau; cuối ô báo số còn lỗi/chưa sinh.
Các file `.partial.wav` không được coi là hoàn thành.

In [ ]:
MANIFEST_FILE = OUTPUT_DIR / 'manifest.json'
INFERENCE_CONFIG = {'model_repo_id': MODEL_REPO_ID, 'model_revision': MODEL_REVISION,
                    'model_name': MODEL_NAME, 'fork_commit': F5_REPO_COMMIT,
                    'speed': SPEED, 'nfe_step': NFE_STEP, 'cfg_strength': CFG_STRENGTH,
                    'sway_sampling_coef': SWAY_SAMPLING_COEF, 'base_seed': BASE_SEED,
                    'pipeline_version': 1}

def new_result(task, ref):
    seed = (BASE_SEED + int(hashlib.sha256(task['task_id'].encode()).hexdigest()[:8], 16)) % (2 ** 31)
    signature = fingerprint({'task_id': task['task_id'], 'text_tts': task['text_tts'],
                             'reference_audio_sha256': ref['audio_sha256'],
                             'reference_text': ref['ref_text_tts'], 'config': INFERENCE_CONFIG, 'seed': seed})
    return {**task, 'voice_requested': VOICE_SPECS[task['scenario_id']],
            'reference': ref, 'seed': seed, 'input_fingerprint': signature, 'status': 'pending'}

def can_resume(row, expected, output_file):
    return (row.get('status') in {'generated', 'resumed'}
            and row.get('input_fingerprint') == expected['input_fingerprint']
            and output_file.is_file() and output_file.stat().st_size > 44
            and file_sha256(output_file) == row.get('output_sha256'))

def run_generation(engine):
    for sid in SELECTED_IDS:
        ref = REFERENCES.get(sid)
        if not ref or ref.get('request_fingerprint') != fingerprint(reference_request(VOICE_SPECS[sid])):
            raise RuntimeError(f'{sid}: chưa có reference đúng cấu hình hiện tại; chạy lại ô chọn reference.')
        if file_sha256(OUTPUT_DIR / ref['ref_audio_relative']) != ref['audio_sha256']:
            raise RuntimeError(f'{sid}: reference WAV đã đổi/hỏng; chạy lại ô chọn reference.')
    old_manifest = json.loads(MANIFEST_FILE.read_text(encoding='utf-8')) if MANIFEST_FILE.is_file() else {}
    previous = old_manifest.get('results', {})
    results = {}
    for task in TASKS:
        row = new_result(task, REFERENCES[task['scenario_id']])
        prior = previous.get(task['task_id'], {})
        if can_resume(prior, row, OUTPUT_DIR / task['output_relative']):
            row.update({key: prior[key] for key in ['sample_rate', 'duration_seconds', 'bytes', 'output_sha256']})
            row['status'] = 'resumed'
        results[task['task_id']] = row
    manifest = {'format_version': 1, 'scenario_ids': SELECTED_IDS, 'input_directory': str(DATASET_DIR),
                'output_directory': str(OUTPUT_DIR), 'expected_turns': len(TASKS),
                'inference_config': INFERENCE_CONFIG,
                'source_licenses': {'vimd': 'CC-BY-NC-ND-4.0', 'checkpoint': 'CC-BY-NC-SA-4.0'},
                'results': results}

    def persist():
        manifest['updated_at_utc'] = datetime.now(timezone.utc).isoformat()
        atomic_json(MANIFEST_FILE, manifest)

    persist()
    selected_tasks = TASKS if MAX_TURNS is None else TASKS[:MAX_TURNS]
    for task in tqdm(selected_tasks, desc='Sinh WAV từng customer_turn'):
        row = results[task['task_id']]
        output_file = OUTPUT_DIR / task['output_relative']
        if row['status'] == 'resumed' and not OVERWRITE:
            continue
        output_file.parent.mkdir(parents=True, exist_ok=True)
        temporary = output_file.with_name(output_file.stem + '.partial.wav')
        temporary.unlink(missing_ok=True)
        ref = row['reference']
        try:
            # Fork tại commit này đọc self.seed; đặt cả hai để seed có hiệu lực.
            engine.seed = row['seed']
            waveform, sample_rate, _ = engine.infer(
                ref_file=str(OUTPUT_DIR / ref['ref_audio_relative']), ref_text=ref['ref_text_tts'],
                gen_text=row['text_tts'], speed=SPEED, nfe_step=NFE_STEP,
                cfg_strength=CFG_STRENGTH, sway_sampling_coef=SWAY_SAMPLING_COEF,
                seed=row['seed'], show_info=lambda message: None,
            )
            waveform = np.asarray(waveform, dtype=np.float32)
            if waveform.ndim != 1 or len(waveform) == 0 or not np.isfinite(waveform).all():
                raise ValueError('Model trả về waveform rỗng/không mono/NaN/Inf.')
            if int(sample_rate) <= 0 or np.max(np.abs(waveform)) < 1e-6:
                raise ValueError('Model trả về sample rate không hợp lệ hoặc toàn im lặng.')
            sf.write(temporary, waveform, int(sample_rate), subtype='PCM_16')
            info = sf.info(temporary)
            if info.frames <= 0 or info.channels != 1:
                raise ValueError('WAV không hợp lệ sau khi ghi.')
            temporary.replace(output_file)
            row.update({'status': 'generated', 'sample_rate': int(info.samplerate),
                        'duration_seconds': float(info.duration), 'bytes': output_file.stat().st_size,
                        'output_sha256': file_sha256(output_file)})
            row.pop('error', None)
        except Exception as exc:
            row['status'] = 'error'
            row['error'] = f'{type(exc).__name__}: {exc}'
            print(f"Lỗi {task['task_id']}: {row['error']}")
        finally:
            temporary.unlink(missing_ok=True)
            persist()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return manifest

if RUN_TTS:
    MANIFEST = run_generation(ENGINE)
    status_counts = pd.Series([row['status'] for row in MANIFEST['results'].values()]).value_counts()
    display(status_counts.rename('turns').to_frame())
    unfinished = [key for key, row in MANIFEST['results'].items() if row['status'] not in {'generated', 'resumed'}]
    if unfinished:
        print(f'Còn {len(unfinished)} lượt lỗi/chưa sinh. Chạy lại sau khi sửa lỗi hoặc đặt MAX_TURNS=None.')
        print(unfinished[:20])
    else:
        print(f'Hoàn thành {len(TASKS)} WAV cho {len(SELECTED_IDS)} scenario.')
    print('Manifest:', MANIFEST_FILE)
else:
    print('Đặt RUN_TTS=True, chạy lại ô nạp model và ô generation để sinh WAV.')

## 10. Kiểm tra đủ file, xuất CSV/JSONL và nghe thử kết quả

Chỉ các file trong manifest của lần chọn scenario/call hiện tại được tính vào kết quả.
`manifest.csv` và `manifest.jsonl` đều có một dòng cho mỗi lượt, kể cả lượt lỗi/chưa sinh;
lọc `status` bằng `generated`/`resumed` khi đưa vào pipeline huấn luyện hoặc đánh giá.
Kiểm tra này xác nhận file và index; độ đúng lời, accent và chất lượng giọng cần nghe/ASR riêng.

In [ ]:
def export_and_check(manifest):
    rows = []
    issues = []
    for key, row in manifest['results'].items():
        ref = row['reference']
        flat = {name: row.get(name) for name in [
            'task_id', 'scenario_id', 'call_id', 'turn_index', 'turn_number',
            'text_original', 'text_tts', 'output_relative', 'status', 'seed',
            'sample_rate', 'duration_seconds', 'bytes', 'output_sha256', 'error']}
        flat.update({'gender_requested': row['voice_requested']['gender_label'],
                     'province_requested': row['voice_requested']['province_requested'],
                     'region_requested': row['voice_requested']['region_requested'],
                     'province_vimd': ref['province_name'], 'region_vimd': ref['region'],
                     'speakerID': ref['speakerID'], 'reference_filename': ref['filename'],
                     'reference_audio_relative': ref['ref_audio_relative'],
                     'reference_text': ref['ref_text_original'], 'reference_split': ref['split']})
        rows.append(flat)
        if row['status'] not in {'generated', 'resumed'}:
            continue
        path = OUTPUT_DIR / row['output_relative']
        try:
            info = sf.info(path)
            if info.frames <= 0 or info.channels != 1 or info.samplerate != row['sample_rate']:
                raise ValueError('WAV header/sample rate không khớp manifest.')
            if file_sha256(path) != row['output_sha256']:
                raise ValueError('Checksum WAV không khớp manifest.')
        except Exception as exc:
            issues.append({'task_id': key, 'error': str(exc)})
    frame = pd.DataFrame(rows)
    if frame.output_relative.duplicated().any() or frame.task_id.duplicated().any():
        raise ValueError('Tên file hoặc task_id bị trùng.')
    frame.to_csv(OUTPUT_DIR / 'manifest.csv', index=False, encoding='utf-8-sig')
    with (OUTPUT_DIR / 'manifest.jsonl').open('w', encoding='utf-8') as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
    atomic_json(OUTPUT_DIR / 'validation_issues.json', issues)
    if issues:
        raise RuntimeError(f'{len(issues)} WAV không khớp manifest; xem validation_issues.json và chạy lại generation.')
    return frame

if RUN_TTS:
    MANIFEST = json.loads(MANIFEST_FILE.read_text(encoding='utf-8'))
    MANIFEST_TABLE = export_and_check(MANIFEST)
    display(MANIFEST_TABLE.groupby('scenario_id').agg(
        expected=('task_id', 'size'),
        completed=('status', lambda values: values.isin(['generated', 'resumed']).sum()),
        seconds=('duration_seconds', 'sum')))
    completed = MANIFEST_TABLE[MANIFEST_TABLE.status.isin(['generated', 'resumed'])]
    for row in completed.head(3).to_dict('records'):
        print(row['output_relative'])
        print('Câu gốc:', row['text_original'])
        print('Câu đọc:', row['text_tts'])
        display(Audio(filename=str(OUTPUT_DIR / row['output_relative'])))

## 11. Đóng gói kết quả (tùy chọn)

Đặt `EXPORT_ZIP=True` rồi chạy ô này. ZIP chỉ chứa WAV thành công trong manifest hiện tại,
reference tương ứng và metadata. Không đưa checkpoint/cache Parquet vào ZIP.
ZIP được lưu trên Drive; nếu còn lượt lỗi/chưa sinh, ô này dừng để tránh xuất bộ thiếu.

In [ ]:
if EXPORT_ZIP:
    import zipfile
    manifest = json.loads(MANIFEST_FILE.read_text(encoding='utf-8'))
    export_and_check(manifest)
    missing = [key for key, row in manifest['results'].items() if row['status'] not in {'generated', 'resumed'}]
    if missing:
        raise RuntimeError(f'Còn {len(missing)} lượt chưa hoàn thành; sinh đủ trước khi export ZIP.')
    paths = {row['output_relative'] for row in manifest['results'].values()}
    paths.update(row['reference']['ref_audio_relative'] for row in manifest['results'].values())
    paths.update(['manifest.json', 'manifest.csv', 'manifest.jsonl', 'validation_issues.json',
                  'references/reference_bank.json', 'references/reference_summary.csv'])
    zip_path = OUTPUT_DIR / 'f5_vimd_customer_turns.zip'
    partial_zip = zip_path.with_name(zip_path.name + '.partial')
    with zipfile.ZipFile(partial_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for relative in sorted(paths):
            archive.write(OUTPUT_DIR / relative, arcname=relative)
    partial_zip.replace(zip_path)
    print('ZIP:', zip_path, f'({zip_path.stat().st_size / 1024 ** 2:.1f} MB)')
else:
    print('Kết quả đang có trên Drive:', OUTPUT_DIR)
    print('Đặt EXPORT_ZIP=True rồi chạy ô này nếu muốn một file ZIP.')